In [1]:
"""
Ovarian CT Classification - Malignancy & Subtype Prediction
===========================================================
Architecture: ResNet50 / ViT / Hybrid (Patient-level Attention Classifier)
Strategy: 5-fold Cross Validation | Cascade (Malignancy -> Subtype)

Phase 2 대비 변경사항:
    [1] Slice 독립 학습 구조
         - 기존: 환자(8 slice) -> mean pool -> 1 vector -> 학습 (444개)
         - 변경: 환자(8 slice) -> 8개 독립 샘플로 학습 (444x8=3552개)
         - SupCon positive pair: 소수 클래스 34개 -> 272개 (8배)

    [2] Patient-level vector 추출 (멀티모달 퓨전 호환)
         - inference: 8 slice -> encoder -> mean pool -> patient vector
         - 나중에 임상 데이터와 concat해서 fusion head에 연결 가능

    [3] 평가 방식
        - 학습: slice 단위 (3552개)
        - 평가: 환자 단위 softmax voting (111명 기준)
        - cascade: malignancy(환자 단위) -> subtype(환자 단위 voting)
"""

'\nOvarian CT Classification - Malignancy & Subtype Prediction\n===========================================================\nArchitecture: ResNet50 / ViT / Hybrid (Patient-level Attention Classifier)\nStrategy: 5-fold Cross Validation | Cascade (Malignancy -> Subtype)\n\nPhase 2 대비 변경사항:\n    [1] Slice 독립 학습 구조\n         - 기존: 환자(8 slice) -> mean pool -> 1 vector -> 학습 (444개)\n         - 변경: 환자(8 slice) -> 8개 독립 샘플로 학습 (444x8=3552개)\n         - SupCon positive pair: 소수 클래스 34개 -> 272개 (8배)\n\n    [2] Patient-level vector 추출 (멀티모달 퓨전 호환)\n         - inference: 8 slice -> encoder -> mean pool -> patient vector\n         - 나중에 임상 데이터와 concat해서 fusion head에 연결 가능\n\n    [3] 평가 방식\n        - 학습: slice 단위 (3552개)\n        - 평가: 환자 단위 softmax voting (111명 기준)\n        - cascade: malignancy(환자 단위) -> subtype(환자 단위 voting)\n'

In [2]:
import os
import random
import numpy as np
from collections import OrderedDict, defaultdict, Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models import ViT_B_16_Weights
import torchvision.models as tv_models
from torchvision import transforms
import torchvision.transforms.functional as TF
from pathlib import Path
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                             f1_score, recall_score, confusion_matrix)
import math
import warnings
warnings.filterwarnings("ignore")

In [3]:
# 0. Config
DATA_ROOT = Path("/tf/nasw/dataset001/preprocessed/npz_ct")
OUTPUT_ROOT = Path("./Cascade_models")

PRETRAINED = {
    "rad_resnet18": Path("/tf/pretrained_model/RadImageNet_resnet18.pth"),
    "rad_resnet50": Path("/tf/pretrained_model/RadImageNet_resnet50.pth"),
    "resnet18": Path("/tf/pretrained_model/resnet18-f37072fd.pth"),
    "resnet50": Path("/tf/pretrained_model/resnet50-11ad3fa6.pth"),
    "vit": Path("/tf/pretrained_model/vit_b_16-c867db91.pth"),
}

FOLDS = [1, 2, 3, 4, 5]
NUM_FOLDS = 5
NUM_SLICES = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 4


# Malignancy 모델 hyperparameter
MAL_BATCH_SIZE = 8
MAL_EPOCHS = 30
MAL_LR_BACKBONE = 1e-5
MAL_LR_HEAD = 5e-5
MAL_WEIGHT_DECAY = 1e-4
MAL_PATIENCE = 12
MAL_MIN_DELTA = 0.001
THRESHOLD = 0.5

# Subtype 모델 hyperparameter
SUB_BATCH_SIZE = 32
SUB_EPOCHS = 100
SUB_LR_BACKBONE = 5e-6
SUB_LR_HEAD = 2e-5
SUB_WEIGHT_DECAY = 1e-4
SUB_PATIENCE = 20
SUB_MIN_DELTA = 0.001
FOCAL_GAMMA = 1.0
LABEL_SMOOTHING = 0.1

# SupCon hyperparameter
SUPCON_WEIGHT = 0.5 # loss = focal + SUPCON_WEIGHT * supcon
SUPCON_TEMP = 0.07 # temperature: 낮을수록 경계가 sharp
PROJ_DIM = 128 # projection head output dim

MALIGNANT_MAP: dict[int, int] = {}
NUM_MALIGNANT_SUBTYPES: int = 0

def infer_malignant_type_ids(data_root, folds):
    label_set_by_type = defaultdict(set)

    for fold_idx in FOLDS:
        for split_name in ["train", "val"]:
            npz_path = Path(data_root) / f"fold_{fold_idx}_{split_name}.npz"
            data = np.load(npz_path, allow_pickle=False)

            labels = data["labels"].astype(int)
            tumor_types = data["tumor_types"].astype(int)

            for y, t in zip(labels, tumor_types):
                label_set_by_type[int(t)].add(int(y))
                
        conflicts = {
            t: sorted(list(v))
            for t, v in label_set_by_type.items()
            if len(v) > 1
        }

        if len(conflicts) > 0:
            raise ValueError(
                f"tumor_type가 양성/악성 양쪽에 섞여있음: {conflicts}\n"
            )
        malignant_type_ids = sorted([
            t for t, labs in label_set_by_type.items()
            if labs == {1}
        ])

        print("malignant_type_ids:", malignant_type_ids)
        return malignant_type_ids


MALIGNANT_TYPE_IDS = infer_malignant_type_ids(DATA_ROOT, FOLDS)
MALIGNANT_MAP = {tid: i for i, tid in enumerate(MALIGNANT_TYPE_IDS)}
NUM_MALIGNANT_SUBTYPES = len(MALIGNANT_TYPE_IDS)
print("NUM_MALIGNANT_SUBTYPES:", NUM_MALIGNANT_SUBTYPES)


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

malignant_type_ids: [0, 1, 3, 5, 7]
NUM_MALIGNANT_SUBTYPES: 5


In [4]:
# 1. Dataset 정의
# online augmentation 추가

class CTAugment:
    def __call__(self, x):
        # Gaussian noise
        if random.random() < 0.5:
            x = x + torch.randn_like(x) * 0.05

        # Random erasing
        if random.random() < 0.3:
            x = transforms.RandomErasing(
                p=1.0, scale=(0.02, 0.08), ratio=(0.3, 3.3)
            )(x)
        return x

# slice 단위 augmentation pipelin
def make_slice_aug():
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        CTAugment(),
    ])

def make_color_aug():
    return transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
    ])
        
        
# 전체 환자 단위 Dataset   
class OvarianCTNPZDataset(Dataset):
    """
    Loads pre-processed CT slices from a .npz file,
    Expected keys: 'images', 'labels', 'tumor_types', 'patients'
    Malignancy 모델 학습 및 cascade 평가에 사용
    """
    def __init__(self, npz_path: Path, augment: bool = False):
        super().__init__()
        self.npz = np.load(npz_path, allow_pickle=False)

        self.images = self.npz["images"]
        self.labels = self.npz["labels"].astype(np.float32)
        self.tumor_types = self.npz["tumor_types"].astype(np.int64)
        self.patient_ids = (self.npz["patient_ids"] if "patient_ids" in self.npz.files 
                            else np.arange(len(self.labels)))

        # RadImageNet normalization constants
        self.mean = torch.tensor([0.204, 0.204, 0.204], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor([0.286, 0.286, 0.286], dtype=torch.float32).view(1, 3, 1, 1)
        
        # Online augmentation (train only)
        self.geo_aug = make_slice_aug() if augment else None
        self.color_aug = make_color_aug() if augment else None


    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx]).float().permute(0, 3, 1, 2).contiguous()
        x = x / 255.0

        # ColorJitter는 [0, 1] 범위에서 적용
        if self.color_aug is not None:
            x = torch.stack([self.color_aug(x[s]) for s in range(x.shape[0])])

        x = (x-self.mean) / self.std

        if self.geo_aug is not None:
            x = torch.stack([self.geo_aug(x[s]) for s in range(x.shape[0])])

        y = self.labels[idx]
        t = int(self.tumor_types[idx])
        mal_subtype = MALIGNANT_MAP[t] if y == 1 else -1
            
        return {
            "image": x, # (S, C, H, W)
            "label": torch.tensor(y, dtype = torch.float32),
            "mal_subtype": torch.tensor(mal_subtype, dtype=torch.long),
            "mal_mask": torch.tensor(mal_mask, dtype=torch.bool),
            "patient_idx": torch.tensor(int(idx), dtype=torch.long),
        }

# Slice 단위 Dataset
class MalignantSliceDataset(Dataset):
    """
    Phase 3: 악성 환자의 slice를 독립 샘플로 펼침

    학습: 각 slice를 독립적인 샘플로 취급
        -> 실질 학습 샘플: 444명 x 8 slice = 3552개
        -> 소수 클래스 0번: 34명 vs 8 = 272개

    평가: 환자 단위 voting (eval_subtype_patient 함수에서 처리)
        -> 8개 slice의 softmax 확률 평균 -> argmax

    멀티모달 퓨전 호환:
        -> extract_patient_vector()로 patient-level vector 추출 가능
        -> clinical vector와 concat 후 fusion head에 연결

    반환 shape:
        "image": (C, H, W) <- 2D slice
        "mal_subtype": scalar
        "patient_idx": scalar <- 환자 단위 voting 시 grouping에 사용
    """
    def __init__(self, npz_path: Path, augment: bool = False):
        super().__init__()
        npz = np.load(npz_path, allow_pickle=False)

        images = npz["images"]
        labels = npz["labels"].astype(np.float32)
        tumor_types = npz["tumor_types"].astype(np.int64)

        # RadImageNet normalization constants
        self.mean = torch.tensor([0.204, 0.204, 0.204], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor([0.286, 0.286, 0.286], dtype=torch.float32).view(1, 3, 1, 1)
        
        # Online augmentation (train only)
        self.geo_aug = make_slice_aug() if augment else None
        self.color_aug = make_color_aug() if augment else None

        # 악성 환자 slice를 독립 샘플로 펼치기
        self.slice_list = [] # (image_np, subtype, patient_idx)

        pat_counter = 0
        counts = Counter()
        for i in range(len(labels)):
            if labels[i] != 1:
                continue
            t = int(tumor_types[i])
            sub = MALIGNANT_MAP[t]
            x = images[i] # (S, H, W, C) numpy

            for s in range(x.shape[0]):
                self.slice_list.append((x[s], sub, pat_counter))
            counts[sub] += 1
            pat_counter += 1

        self.num_patients = pat_counter
        print(f"  [SliceDataset]"
              f"환자 {self.num_patients}명 x {NUM_SLICES} slice = "
              f"{len(self.slice_list)}개 |"
              f"subtype 분포: {[counts[k] for k in range(NUM_MALIGNANT_SUBTYPES)]}"
              f" -> slice 기준: {[counts[k]*NUM_SLICES for k in range(NUM_MALIGNANT_SUBTYPES)]}")

    def __len__(self):
        return len(self.slice_list)

    def __getitem__(self, idx):
        slice_np, sub, pat_idx = self.slice_list[idx]

        # (H, W, C) -> (C, H, W), normalize to [0,1]
        x = torch.from_numpy(slice_np).float().permute(2, 0, 1).contiguous()
        x = x / 255.0

        # ColorJitter는 [0, 1] 범위에서 적용
        if self.color_aug is not None:
            x = self.color_aug(x)
        
        x = (x-self.mean) / self.std

        if self.geo_aug is not None:
            x = self.geo_aug(x)

        if x.ndim == 4:
            x = x.squeeze(0) # (1, C, H, W) -> (C, H, W)
            
        return {
            "image": x, # (C, H, W)
            "mal_subtype": torch.tensor(sub, dtype=torch.long),
            "patient_idx": torch.tensor(pat_idx, dtype=torch.long),
        }

    def get_ens_weights(self) -> torch.Tensor:
        counts = np.array([
        sum(1 for _, sub, _ in self.slice_list if sub == c) / NUM_SLICES
            for c in range(NUM_MALIGNANT_SUBTYPES)
        ], dtype=np.float64)
        counts = np.maximum(counts, 1.0)
        beta = 0.9999
        effective_num = (1.0 - np.power(beta, counts)) / (1.0 - beta)
        weights_ens = 1.0 / effective_num
        weights_ens = weights_ens / weights_ens.min()
        subtype_class_weights = torch.tensor(weights_ens, dtype=torch.float32, device=DEVICE)
        print(f"  subtype_class_weights={weights_ens.round(4).tolist()}")
        
        return subtype_class_weights


# Patient-level val Dataset (subtype 평가용)
class MalignantPatientDataset(Dataset):
    """
    평가 전용: 환자 단위로 8개 slice를 묶어서 반환
    train은 MalignantSliceDataset, val 평가는 이 클래스 사용
    반환 shape: "image": (S, C, H, W)
    """
    def __init__(self, npz_path: Path, augment: bool = False):
        super().__init__()
        npz = np.load(npz_path, allow_pickle=False)

        images = npz["images"]
        labels = npz["labels"].astype(np.float32)
        tumor_types = npz["tumor_types"].astype(np.int64)

        # RadImageNet normalization constants
        self.mean = torch.tensor([0.204, 0.204, 0.204], dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor([0.286, 0.286, 0.286], dtype=torch.float32).view(1, 3, 1, 1)

        self.patient_data = []
        counts = Counter()

        for i in range(len(labels)):
            if labels[i] != 1:
                continue
            t = int(tumor_types[i])
            sub = MALIGNANT_MAP[t]
            self.patient_data.append((images[i], sub, len(self.patient_data)))
            counts[sub] += 1
            
        print(f"  [PatientDataset] 총 {len(self.patient_data)}명 | "
              f"subtype 분포: {[counts[k] for k in range(NUM_MALIGNANT_SUBTYPES)]}")

    def __len__(self):
        return len(self.patient_data)

    def __getitem__(self, idx):
        x_np, sub, pat_idx = self.patient_data[idx]

        # (S, H, W, C) -> (S, C, H, W), normalize to [0,1] then ImageNet-nomalize
        x = torch.from_numpy(x_np).float().permute(0, 3, 1, 2).contiguous()
        x = x / 255.0
        x = (x-self.mean) / self.std
        
        return {
            "image": x, # (S, C, H, W)
            "mal_subtype": torch.tensor(sub, dtype=torch.long),
            "patient_idx": torch.tensor(pat_idx, dtype=torch.long),
        }



In [5]:
# 2. Encoders
def _load_local_weights(model: nn.Module, path: Path, strict: bool = True) -> nn.Module:
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state, strict=strict)
    return model

def _load_rad_imagenet_weights(model: nn.Module, path: Path) -> nn.Module:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    if isinstance(ckpt, dict):
        if "model" in ckpt:
            state = ckpt["model"]
        elif "state_dict" in ckpt:
            state = ckpt["state_dict"]
        else:
            state = ckpt

    else:
        raise ValueError(f"Unexpected checkpoint type: {type(ckpt)}")

    # _orig_mod. prefix 제거
    state = {k.replace("_orig_mod.", ""): v 
             for k, v in state.items()
            }

    # fc layer 제외
    state = {k: v for k, v in state.items() if not k.startswith("fc.")}

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"  [RadImageNet] loaded {path.name}")
    if missing:
        print(f"   missing keys ({len(missing)}): {missing[:3]}{'...' if len(missing) > 3 else ''}")
    if unexpected:
        print(f"   unexpected keys ({len(unexpected)}): {unexpected[:3]}{'...' if len(unexpected) > 3 else ''}")

    return model
    
def build_encoder(freeze_backbone: bool = True, 
                  unfreeze_layer4: bool = True,
                  unfreeze_layer3: bool = False) -> tuple[nn.Module, int]:
    
    """
    RadImageNet ResNet18 encoder
    """
    encoder = tv_models.resnet18(weights=None)
    encoder = _load_rad_imagenet_weights(encoder, PRETRAINED["rad_resnet18"])
    feat_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()

    if freeze_backbone:
        for p in encoder.parameters():
            p.requires_grad = False
        if unfreeze_layer4:
            for p in encoder.layer4.parameters():
                p.requires_grad = True
        if unfreeze_layer3:
            for p in encoder.layer3.parameters():
                p.requires_grad = True
    
    return encoder, feat_dim

In [6]:
# 3. Classifers

class MalignancyModel(nn.Module):
    """
    Malignancy 이진 분류 모델
    Attention pooling -> shared head -> binary output.
    """
    def __init__(self, encoder: nn.Module, feat_dim: int, 
                 hidden_dim: int = 256,
                 dropout: float = 0.3):
        super().__init__()
        self.encoder = encoder
        
        # Attention pooling: feat_dim -> feat_dim//2 -> 1
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.Tanh(),
            nn.Linear(feat_dim // 2, 1)
        )

        self.head = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        # x: (B, S, C, H, W)
        B, S, C, H, W = x.shape

        # Encode all slices in parallel
        feat = self.encoder(x.view(B * S, C, H, W)).view(B, S, -1) # (B, S, feat_dim)
        
        # Attention-weighted patient-level pooling
        attn_weight = torch.softmax(self.attn(feat), dim=1) # (B, S, 1)
        pooled = (feat * attn_weight).sum(dim=1) # (B, feate_dim)
        
        return self.head(pooled).squeeze(1)



class SubtypeSliceModel(nn.Module):
    """
    Phase 3: slice 단위 분류 모델

    학습 입력: (B, C, H, W) - 개별 slice
    추론 입력: (S, C, H, W) - 환자의 8개 slice -> voting
    """
    def __init__(self, encoder: nn.Module, feat_dim: int, 
                 num_subtypes: int = NUM_MALIGNANT_SUBTYPES,
                 hidden_dim: int = 256, dropout: float = 0.7,
                 proj_dim: int = PROJ_DIM):
        super().__init__()
        self.encoder = encoder
        self.feat_dim = feat_dim
        self.hidden_dim = hidden_dim
    
        # Classification head (Focal Loss용)
        self.cls_head = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_subtypes)
        )

        # Projection head (SupCon Loss용)
        # 2-layer MLP: feat_dim -> feat_dim -> proj_dim
        # 학습 후 버리고 cls_head만 inference에 사용
        self.proj_head = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(),
            nn.Linear(feat_dim, proj_dim)
        )

    def forward(self, x):
        # x: (B, S, C, H, W) - 개별 slice (학습 시)
        # or (S, C, H, W) - 환자 8 slice (inference 시)

        # Encode all slices in parallel
        feat = self.encoder(x) # (B, feat_dim)
        logits = self.cls_head(feat) # (B, num_subtypes)
        proj = F.normalize(self.proj_head(feat), dim=1) # (B, proj_dim) L2 정규화
        
        return logits, proj

In [7]:
# 4. Loss

class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss.
        FL(p_t) = -alpha_t * (1-p_t)^gamma * log(p_t)
    - alpha (class_weight): 다수 클래스 억제 (ENS weight 그대로 사용)
    - gamma: easy example 억제 (기본값 2.0)
    - smoothing: 정답 타겟을 1.0 -> (1-smoothing)으로 낮춰 과적합 억제
    """
    def __init__(self, weight: torch.Tensor, gamma: float = 2.0,
                 smoothing: float = 0.0):
        super().__init__()
        self.register_buffer("weight", weight.float())
        self.gamma = gamma
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        num_classes = logits.size(1)
        log_prob = nn.functional.log_softmax(logits, dim=1) # (N, C)
        prob = log_prob.exp() # (N, C)

        # alpha: 각 샘플의 클래스 weight
        alpha = self.weight[targets] # (N,)

        # p_t: 정답 클래스의 확률
        p_t = prob[torch.arange(len(targets)), targets] # (N,)

        # Focal weight
        focal_w = (1.0 - p_t) ** self.gamma # (N,)

        if self.smoothing > 0.0:
            # soft target: 정답=1-s, 나머지=s/(C-1)
            smooth_val = self.smoothing / (num_classes - 1)
            soft_target = torch.full_like(log_prob, smooth_val)
            soft_target.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
            loss_per_sample = -(soft_target * log_prob).sum(dim=1)
        else:
            loss_per_sample = -log_prob[torch.arange(len(targets)), targets]

        loss = (alpha * focal_w * loss_per_sample).mean()
        return loss

class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., 2020)

    핵심 아이디어:
        slice 단위 학습에서 같은 subtype의 샘플들은 feature space에서 가깝게,
        다른 subtype은 멀게 밀어내는 방식으로 학습,
        소수 클래스(34개)가 다수 클래스(249개)에 묻히는 문제를 직접 공략
        numerical stability를 위해 exp + clamp 방식 사용

    주의:
    - 배치 내에 같은 클래스가 2개 이상 있어야 loss 계산 가능
    - 배치 크기가 작으면 (8) positive pair가 없는 클래스 발생 가능
        -> 해당 샘플은 loss 계산에서 자동으로 제외됨
    """
    def __init__(self, temperature: float = SUPCON_TEMP):
        super().__init__()
        self.temperature = temperature

    def forward(self, features: torch.Tensor,
                labels: torch.Tensor) -> torch.Tensor:
        """
        Args
        ----
            features: L2 정규화된 projection feature (N, proj_dim)
            labels: subtype 레이블 (N,)

        Returns
        -------
            scalar loss
        """
        
        N = features.size(0)
        device = features.device

        # Cosine similarity matrix (N, N) -> temperature scaling
        sim_matrix = torch.matmul(features, features.T) / self.temperature

        # 자기 자신과의 유사도 제거 (대각선 마스크)
        self_mask = torch.eye(N, dtype=torch.bool, device=device)
        sim_matrix = sim_matrix.masked_fill(self_mask, float("-inf"))

        # Positive mask: 같은 클래스인 쌍 (자기 자신 제외)
        labels = labels.view(-1, 1)
        pos_mask = (labels == labels.T) * ~self_mask # (N, N) bool

        # Positive pair가 없는 샘플 제외
        # (배치 내 해당 클래스가 1개밖에 없는 경우)
        valid_mask = pos_mask.sum(dim=1) > 0 # (N,)
        if valid_mask.sum() == 0:
            # 배치 전체에서 positive pair가 없으면 loss = 0
            return torch.tensor(0.0, device=device, requires_grad=True)

        # log_softmax 대신 수동으로 계산해서 numerical stability 확보
        # -inf 대신 자기 자신을 exp에서 제외하는 방식으로 변경
        exp_sim = torch.exp(sim_matrix) * (~self_mask).float() # 자기 자신 제외

        # 분모: 자기 자신을 제외한 모든 샘플과의 유사도 합
        denom = exp_sim.sum(dim=1, keepdim=True).clamp(min=1e-9) # (N, 1)

        # log prob
        log_prob = torch.log(exp_sim / denom + 1e-9) # (N, N)

        # 각 anchor에서 positive pair의 log_prob 평균
        pos_log_prob = (pos_mask * log_prob).sum(dim=1) # (N,)
        num_positives = pos_mask.sum(dim=1).float() # (N,)

        # valid_mask로 positive pair 없는 샘플 제외
        loss = -(pos_log_prob[valid_mask] / num_positives[valid_mask]).mean()
        
        return loss

In [8]:
# 5. DataLoader factory (per fold)
def make_malignancy_loaders(fold_idx: int) -> tuple:
    train_npz = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz = DATA_ROOT / f"fold_{fold_idx}_val.npz"
    assert train_npz.exists(), f"Missing: {train_npz}"
    assert val_npz.exists(), f"Missing: {val_npz}"

    train_dataset = OvarianCTNPZDataset(train_npz, augment=True)
    val_dataset = OvarianCTNPZDataset(val_npz, augment=False)

    train_loader = DataLoader(train_dataset, batch_size=MAL_BATCH_SIZE,
                              shuffle = True, num_workers = NUM_WORKERS, pin_memory = True)
    val_loader = DataLoader(val_dataset, batch_size=MAL_BATCH_SIZE, 
                            shuffle = False,num_workers = NUM_WORKERS, pin_memory = True)

    neg = (train_dataset.labels == 0).sum()
    pos = (train_dataset.labels == 1).sum()
    pos_weight = torch.tensor([neg / max(pos, 1)], device=DEVICE,
                              dtype=torch.float32)

    print(f"[FOLD {fold_idx}] "
          f"benign(0)={(train_dataset.labels==0).sum()} "
          f"malignant(1)={(train_dataset.labels==1).sum()} "
          f"pos_weight={pos_weight.item():.4f}")

    return train_loader, val_loader, pos_weight

def make_subtype_loaders(fold_idx: int) -> tuple:
    train_npz = DATA_ROOT / f"fold_{fold_idx}_train.npz"
    val_npz = DATA_ROOT / f"fold_{fold_idx}_val.npz"
    assert train_npz.exists(), f"Missing: {train_npz}"
    assert val_npz.exists(), f"Missing: {val_npz}"

    train_dataset = MalignantSliceDataset(train_npz, augment=True)
    val_dataset = MalignantPatientDataset(val_npz, augment=False)

    
    train_loader = DataLoader(train_dataset, batch_size=SUB_BATCH_SIZE,
                              shuffle = True, num_workers = NUM_WORKERS, pin_memory = True)
    val_loader = DataLoader(val_dataset, batch_size=1, 
                            shuffle = False, num_workers = NUM_WORKERS, pin_memory = True)

    subtype_class_weights = train_dataset.get_ens_weights()

    return train_loader, val_loader, subtype_class_weights

In [9]:
# 6. metrics

def _compute_binary_metrics(labels: list, probs: list,
                            threshold: float = THRESHOLD) -> dict:
    """
    ACC, AUC, F1, Recall, Confusion Matrix for malignancy.
    """
    preds = (np.array(probs) >= threshold).astype(int)
    labs = np.array(labels).astype(int)
    auc = roc_auc_score(labs, probs) if len(set(labs)) > 1 else float("nan")

    return {
        "acc": float(accuracy_score(labs, preds)),
        "auc": float(auc),
        "f1": float(f1_score(labs, preds, zero_division=0)),
        "recall": float(recall_score(labs, preds, zero_division=0)),
        "cm": confusion_matrix(labs, preds).tolist(),
    }

def _compute_multiclass_metrics(true: np.ndarray, preds: np.ndarray) -> dict:
    """
    ACC, marco-F1, Confusion Matrix for subtype.
    """
    return {
        "acc": float(accuracy_score(true, preds)),
        "macro_f1": float(f1_score(true, preds, average="macro", zero_division=0)),
        "cm": confusion_matrix(true, preds).tolist(),
    }

In [10]:
# 7. Train / Eval: Malignancy
def train_malignancy_epoch(model: nn.Module, loader: DataLoader, 
                           criterion: nn.BCEWithLogitsLoss, 
                           optimizer, scaler) -> dict:
    model.train()
    running_loss = 0.0
    all_label, all_prob = [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE).view(-1)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)

            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        all_label.extend(labels.detach().cpu().numpy().astype(int).tolist())
        all_prob.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())

    m = _compute_binary_metrics(all_label, all_prob)
    m["loss"] = running_loss / len(loader.dataset)

    return m


@torch.no_grad()
def eval_malignancy_epoch(model: nn.Module, loader: DataLoader, 
                         criterion: nn.BCEWithLogitsLoss) -> dict:
    model.eval()
    running_loss = 0.0
    all_label, all_prob = [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE).view(-1)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            outputs = model(imgs)

            logits = model(imgs)
            loss = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)
        all_label.extend(labels.detach().cpu().numpy().astype(int).tolist())
        all_prob.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())

    m = _compute_binary_metrics(all_label, all_prob)
    m["loss"] = running_loss / len(loader.dataset)

    return m

In [11]:
# 8. Train / Eval: Subtype
def train_subtype_epoch(model: nn.Module, 
                        loader: DataLoader, 
                        focal_criterion: FocalLoss, 
                        supcon_criterion: SupConLoss,
                        optimizer, 
                        scaler,
                        supcon_weight: float = SUPCON_WEIGHT) -> dict:
    """
    slice 단위 학습
    입력: (B, C, H, W) - 개별 slice
    """
    model.train()
    running_loss = running_focal = running_supcon = 0.0
    all_true, all_pred = [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        subtypes = batch["mal_subtype"].to(DEVICE).view(-1)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits, proj = model(imgs)

            loss_focal = focal_criterion(logits, subtypes)
            loss_supcon = supcon_criterion(proj, subtypes)

            # SupConLoss가 0이면 (positive pair 없음) focal만 사용
            loss = loss_focal + supcon_weight * loss_supcon

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = imgs.size(0)
        running_loss += loss.item() * bs
        running_focal += loss_focal.item() * bs
        running_supcon += loss_supcon.item() * bs
        
        all_true.extend(subtypes.cpu().numpy().tolist())
        all_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())

    n = len(loader.dataset)
    m = _compute_multiclass_metrics(np.array(all_true), np.array(all_pred))
    m["loss"] = running_loss / n
    m["loss_focal"] = running_focal / n
    m["loss_supcon"] = running_supcon / n

    return m


@torch.no_grad()
def eval_subtype_epoch(model: nn.Module, loader: DataLoader, 
                       focal_criterion: FocalLoss) -> dict:
    """ 
    환자 단위 평가: 8개의 slice의 softmax 확률 평균 -> argmax.
    val_loader는 batch_size=1 (환자 1명씩).
    """
    model.eval()
    running_loss = 0.0
    all_true, all_pred = [], []

    for batch in loader:
        # batch_size=1이므로 squeeze(0) -> (S, C, H, W)
        imgs = batch["image"].squeeze(0).to(DEVICE, non_blocking=True)
        subtypes = batch["mal_subtype"].to(DEVICE).view(-1)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits, _ = model(imgs) # (S, num_subtypes)
            # 환자 단위 loss: 평균 logits으로 계산
            avg_logit = logits.mean(dim=0, keepdim=True) # (1, num_subtypes)
            loss = focal_criterion(avg_logit, subtypes)

        # Softmax voting: S개 slice 확률 평균 -> argmax
        probs = F.softmax(logits, dim=1)
        avg_prob = probs.mean(dim=0)
        pred = int(avg_prob.argmax())

        running_loss += loss.item()
        all_true.append(int(subtypes.cpu()))
        all_pred.append(pred)

    m = _compute_multiclass_metrics(np.array(all_true), np.array(all_pred))
    m["loss"] = running_loss / len(loader.dataset)

    return m

In [12]:
# 9. Cascade Evaludation

@torch.no_grad()
def evaluate_cascade(mal_model: MalignancyModel,
                     sub_model: SubtypeSliceModel,
                     loader: DataLoader) -> dict:
    """
    실제 cascade 추론:
        1단계: malignancy 예측 (환자 단위)
        2단계: Subtype 예측 (8 slice softmax voting)
    """
    mal_model.eval()
    sub_model.eval()

    all_label, all_mal_prob, all_mal_pred = [], [], []
    all_subtype_true, all_subtype_pred = [], []
    all_mal_mask = []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].cpu().numpy().astype(int).ravel()
        mal_mask = batch["mal_mask"].cpu().numpy().astype(bool).ravel()
        sub_true = batch["mal_subtype"].cpu().numpy().astype(int).ravel()

        B, S, C, H, W = imgs.shape
        

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            # 1단계: malignancy
            mal_logits = mal_model(imgs)

            # 2단계: subtype (각 환자의 S개 slice -> voting)
            sub_preds = []
            for b in range(B):
                slices = imgs[b]
                slice_logits, _ = sub_model(slices)
                avg_prob = F.softmax(slice_logits, dim=1).mean(dim=0)
                sub_preds.append(int(avg_prob.argmax()))
            
        mal_prob = torch.sigmoid(mal_logits).cpu().numpy().ravel()
        mal_pred = (mal_prob >= TRHESHOLD).astype(int)

        all_label.extend(labels.tolist())
        all_mal_prob.extend(mal_prob.tolist())
        all_mal_pred.extend(mal_pred.toslit())
        all_mal_mask.extend(mal_mask.tolist())
        all_subtype_true.extend(sub_true.tolist())
        all_subtype_pred.extend(sub_pred.tolist())

    all_label = np.array(all_label).astype(int)
    all_mal_pred = np.array(all_mal_pred).astype(int)
    all_mal_mask = np.array(all_mal_mask).astype(bool)
    all_sub_true = np.array(all_subtype_true).astype(int)
    all_sub_pred = np.array(all_subtype_pred).astype(int)

    # Malignancy metrics
    mal_m = _compute_binary_metrics(all_label.toslit(), all_mal_prob)

    # Oracle subtype (GT 악성 기준, 1단계 무시)
    oracle_true = all_sub_true[all_mal_mask]
    oracle_pred = all_sub_pred[all_mal_mask]
    oracle_m = _compute_multiclass_metrics(oracel_true, oracle_pred)

    # Cascade subtype (실제 cascade 기준)
    gt_mal_mask = (all_label == 1)
    if get_mal_mask.sum() > 0:
        gate_recall = (all_mal_pred[gt_mal_mask] == 1).mean()
        tp_mask = (all_mal_pred[gt_mal_mask] == 1)
        tp_sub_true = oracle_true[tp_mask]
        tp_sub_pred = oracle_pred[tp_mask]
        cascade_acc = (
            (tp_sub_true == tp_sub_pred).sum() / gt_mal_mask.sum()
            if tp_mask.sum() > 0 else 0.0
        )
    else:
        gate_recall = cascade_acc = np.nan

    print(f"  [Cascade Eval]"
          f"  mal_auc={mal_m['auc']:.4f}"
          f"  mal_f1={mal_m['f1']:.4f}"
          f"  oracle_acc={oracle_m['acc']:.4f}"
          f"  oracle_f1={oracle_m['f1']:.4f}"
          f"  cascade_acc={float(cascade_acc):.4f}"
          f"  gate_recall={float(gate_recall):.4f}")

    return {
        "mal_auc": mal_m['auc'],
        "mal_f1": mal_m['f1'],
        "mal_recall": mal_m['recall'],
        "mal_cm": mal_m['cm'],
        "oracle_acc": oracle_m["acc"],
        "oracle_macro_f1": oracle_m["macro_f1"],
        "oracle_cm": oracle_m["cm"],
        "cascade_cm": float(cascade_acc),
        "gate_recall": float(gate_recall),
    }


In [13]:
# 10. Step 1: Malignancy 모델 학습

def train_malignancy_model(fold_idx: int) -> MalignancyModel:
    print(f"\n[Step 1] Malignancy | Fold {fold_idx}/{NUM_FOLDS}")

    train_loader, val_loader, pos_weight = make_malignancy_loaders(fold_idx)

    encoder, feat_dim = build_encoder(freeze_backbone=True,
                                      unfreeze_layer4=True)
    model = MalignancyModel(encoder, feat_dim).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        [{"params": [p for n, p in model.named_parameters()
                       if p.requires_grad and "encoder" in n], "lr": MAL_LR_BACKBONE},
         {"params": [p for n, p in model.named_parameters()
                   if p.requires_grad and "encoder" not in n], "lr": MAL_LR_HEAD}],
        weight_decay=MAL_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAL_EPOCHS, eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_auc, best_state, patience_counter = 0.0, None, 0

    for epoch in range(1, MAL_EPOCHS + 1):
        tr = train_malignancy_epoch(model, train_loader, criterion,
                                    optimizer, scaler)
        vl = eval_malignancy_epoch(model, val_loader, criterion)
        scheduler.step()

        print(f"  Epoch {epoch:02d}/{MAL_EPOCHS} "
              f"loss={tr['loss']:.4f}|{vl['loss']:.4f} "
              f"auc={tr['auc']:.4f}|{vl['auc']:.4f} "
              f"f1={tr['f1']:.4f}|{vl['f1']:.4f} "
              f"patience={patience_counter}|{MAL_PATIENCE} ")

        if vl["auc"] > best_auc + MAL_MIN_DELTA:
            best_auc = vl["auc"]
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= MAL_PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    torch.save(best_state,
               OUTPUT_ROOT / f"malignancy_fold{fold_idx}_best.pth")
    print(f"  Best val AUC: {best_auc:.4f}")
    return model


In [19]:
# 11. Step 2: Subtype 모델 학습

def train_subtype_model(fold_idx: int) -> SubtypeSliceModel:
    print(f"\n[Step 2] Subtype | Fold {fold_idx}/{NUM_FOLDS}")

    train_loader, val_loader, subtype_weights = make_subtype_loaders(fold_idx)

    encoder, feat_dim = build_encoder(freeze_backbone=True,
                                      unfreeze_layer4=True,
                                      unfreeze_layer3=True)
    model = SubtypeSliceModel(encoder, feat_dim).to(DEVICE)
    focal_crit = FocalLoss(weight=subtype_weights,
                          gamma=FOCAL_GAMMA,
                          smoothing=LABEL_SMOOTHING)
    supcon_crit = SupConLoss(temperature=SUPCON_TEMP)
    
    optimizer = torch.optim.AdamW(
        [{"params": [p for n, p in model.named_parameters()
                       if p.requires_grad and "encoder" in n], "lr": SUB_LR_BACKBONE},
         {"params": [p for n, p in model.named_parameters()
                   if p.requires_grad and "encoder" not in n], "lr": SUB_LR_HEAD}],
        weight_decay=SUB_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAL_EPOCHS, eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    best_f1, best_state, patience_counter = 0.0, None, 0

    for epoch in range(1, SUB_EPOCHS + 1):
        tr = train_subtype_epoch(model, train_loader, focal_crit,
                                 supcon_crit, optimizer, scaler)
        vl = eval_subtype_epoch(model, val_loader, focal_crit)
        scheduler.step()

        print(f"  Epoch {epoch:02d}/{SUB_EPOCHS} "
              f"loss={tr['loss']:.4f}|{vl['loss']:.4f} "
              f"(focal={tr['loss_focal']:.4f} supcon={tr['loss_supcon']:.4f}) "
              f"acc={tr['acc']:.4f}|{vl['acc']:.4f} "
              f"f1={tr['macro_f1']:.4f}|{vl['macro_f1']:.4f} "
              f"patience={patience_counter}|{SUB_PATIENCE} ")

        if vl["macro_f1"] > best_f1 + SUB_MIN_DELTA:
            best_f1 = vl["macro_f1"]
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= SUB_PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    torch.save(best_state,
               OUTPUT_ROOT / f"subtype_fold{fold_idx}_best.pth")
    print(f"  Best val macro_F1: {best_f1:.4f}")
    return model


In [15]:
#12. Cascade 평가

def load_malignancy_model(fold_idx: int) -> nn.Module:
    encoder, feat_dim = build_encoder(freeze_backbone=True)
    model = MalignancyModel(encoder, feat_dim).to(DEVICE)
    state = torch.load(OUTPUT_ROOT / f"malignancy_fold{fold_idx}_best.pth",
                       map_location=DEVICE)
    model.load_state_dict(state)
    return model

def load_subtype_model(fold_idx: int) -> nn.Module:
    encoder, feat_dim = build_encoder(freeze_backbone=True)
    model = SubtypeModel(encoder, feat_dim).to(DEVICE)
    state = torch.load(OUTPUT_ROOT / f"subtype_fold{fold_idx}_best.pth",
                       map_location=DEVICE)
    model.load_state_dict(state)
    return model

In [16]:
# 13. Full CV experiment

def run_cv_experiment():
    print(f"\n{'='*60}")
    print(f"  Phase 3 (Slice 독립 + SupCon) | Device: {DEVICE}")
    print(f"{'='*60}")
    
    fold_results = []

    for fold_idx in range(1, NUM_FOLDS + 1):
        print(f"\n{'-'*50}")
        print(f" FOLD {fold_idx}/{NUM_FOLDS}")
        print(f"{'-'*50}")
        
        # Step 1: Malignancy 모델 학습
        mal_model = train_malignancy_model(fold_idx)

        # Step 2: Subtype 모델 학습
        sub_model = train_subtype_model(fold_idx)

        # Step 3: Cascade 평가 (val set 기준)
        print(f"\n[Step 3] Cascade Evaluation | Fold {fold_idx}")
        _, val_loader, _ = mak_malignancy_loaders(fold_idx)
        result = evaluate_cascade(mal_model, sub_model, val_loader)
        fold_results.append(result)

    # CV 결과 요약
    print(f"\n{'='*60}")
    print(f"  5-Fold CV Summary")
    print(f"{'='*60}")

    for key in ["mal_auc", "mal_f1", "oracle_acc", "oracle_macro_f1",
                "cascade_acc", "gate_recall"]:
        vals = [r[key] for r in fold_results
                if not np.isnan(r[key])]
        print(f"  {key:25s}: {np.mean(vals):.4f} +- {np.std(vals):.4f}")

    return fold_results

In [17]:
# 14. Entry point

#if __name__ == "__main__":
#    results = run_cv_experiment()

In [20]:
for fold_idx in range(1, NUM_FOLDS + 1):
    print(f"\n{'-'*50}")
    print(f" FOLD {fold_idx}/{NUM_FOLDS}")
    print(f"{'-'*50}")
        
    # Step 2: Subtype 모델 학습
    sub_model = train_subtype_model(fold_idx)


--------------------------------------------------
 FOLD 1/5
--------------------------------------------------

[Step 2] Subtype | Fold 1/5
  [SliceDataset]환자 444명 x 8 slice = 3552개 |subtype 분포: [34, 67, 48, 46, 249] -> slice 기준: [272, 536, 384, 368, 1992]
  [PatientDataset] 총 111명 | subtype 분포: [8, 17, 12, 12, 62]
  subtype_class_weights=[7.2454, 3.6828, 5.1358, 5.3585, 1.0]
  [RadImageNet] loaded RadImageNet_resnet18.pth
   missing keys (2): ['fc.weight', 'fc.bias']
  Epoch 01/100 loss=5.8070|3.4952 (focal=4.0903 supcon=3.4336) acc=0.2061|0.2793 f1=0.1783|0.2277 patience=0|20 
  Epoch 02/100 loss=5.2639|3.5301 (focal=3.5507 supcon=3.4264) acc=0.2142|0.1532 f1=0.1818|0.1396 patience=0|20 
  Epoch 03/100 loss=5.2835|3.5814 (focal=3.5740 supcon=3.4191) acc=0.2463|0.0721 f1=0.1413|0.0269 patience=1|20 
  Epoch 04/100 loss=5.2742|3.5191 (focal=3.5680 supcon=3.4124) acc=0.1757|0.5856 f1=0.1489|0.2593 patience=2|20 
  Epoch 05/100 loss=5.2876|3.5750 (focal=3.5828 supcon=3.4095) acc=0.3097

Exception in thread Thread-138 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/pin_memory.py", line 61, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/pin_memory.py", line 37, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/multiprocessing/reductions.py"

KeyboardInterrupt: 